In [2]:
import polars as pl 
import polars.selectors as cs
from datetime import datetime
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from time import sleep
from src.utils import (
    scrape_eliteserien_all_xg_for_seasons,
    scrape_eliteserien_results_for_seasons,
    scrape_eliteserien_general_player_statistics_for_seasons
)


In [10]:
seasons = [
    (2020,2021),
    (2021,2022),
    (2022,2023),
    (2023,2024),
    (2024,2025),
    (2025,2026)
]

db_path = Path("data/brann.duckdb")

In [10]:
def save_eliteserien_results_for_season(
    season: tuple[int, int | None],
    db_path: Path,
    delay_seconds: float = 0.5,
) -> None:
    """Scrape one Eliteserien season and store it in its own raw_eliteserien_results_<year> table."""
    year = season[1] or season[0]
    table_name = f"raw_eliteserien_results_{year}"

    data = scrape_eliteserien_results_for_seasons([season], delay_seconds=delay_seconds)
    if not data:
        print(f"No records returned for season {season}; skipping.")
        return

    df = pl.DataFrame(data).with_columns(pl.lit(datetime.now()).alias("ingested_at"))

    with duckdb.connect(str(db_path)) as connection:
        connection.register("new_data", df)
        connection.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM new_data")

    print(f"✓ Saved {len(data)} matches to {table_name}")


# Ingest results season by season so one failing season doesn't block the rest
for season in seasons:
    try:
        save_eliteserien_results_for_season(season, db_path, delay_seconds=0.5)
    except Exception as error:
        print(f"Skipping season {season}: {error}")


✓ Saved 240 matches to raw_eliteserien_results_2021
✓ Saved 240 matches to raw_eliteserien_results_2022
✓ Saved 240 matches to raw_eliteserien_results_2023
✓ Saved 240 matches to raw_eliteserien_results_2024
✓ Saved 240 matches to raw_eliteserien_results_2025
✓ Saved 168 matches to raw_eliteserien_results_2026


In [20]:
def save_eliteserien_xg_for_season(
    season: tuple[int, int | None],
    db_path: Path,
    delay_seconds: float = 0.01,
    season_delay_seconds: float = 0.1,
    timeout_seconds: int = 30,
) -> None:
    """Scrape one Eliteserien season's xG/stats and store it in its own raw_match_statistics_<year> table."""
    year = season[1] or season[0]
    table_name = f"raw_match_statistics_{year}"

    data = scrape_eliteserien_all_xg_for_seasons(
        [season],
        delay_seconds=delay_seconds,
        season_delay_seconds=season_delay_seconds,
        timeout_seconds=timeout_seconds,
    )
    if not data:
        print(f"No records returned for season {season}; skipping.")
        return

    df = pl.DataFrame(data).with_columns(pl.lit(datetime.now()).alias("ingested_at"))

    with duckdb.connect(str(db_path)) as connection:
        connection.register("new_data", df)
        connection.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM new_data")

    print(f"✓ Saved {len(data)} matches to {table_name}")


# Ingest xG/stats season by season, waiting 10s between seasons so one failing season doesn't block the rest
for index, season in enumerate(seasons):
    if index:
        sleep(10)
    try:
        save_eliteserien_xg_for_season(season, db_path)
    except Exception as error:
        print(f"Skipping season {season}: {error}")


Fetching xG for season 2021 (ID: 2020)...
✓ Saved 240 matches to raw_match_statistics_2021
Fetching xG for season 2022 (ID: 2021)...
✓ Saved 240 matches to raw_match_statistics_2022
Fetching xG for season 2023 (ID: 2022)...
✓ Saved 240 matches to raw_match_statistics_2023
Fetching xG for season 2024 (ID: 2023)...
✓ Saved 240 matches to raw_match_statistics_2024
Fetching xG for season 2025 (ID: 2024)...
✓ Saved 240 matches to raw_match_statistics_2025
Fetching xG for season 2026 (ID: 2025)...
✓ Saved 168 matches to raw_match_statistics_2026


In [23]:
from src.utils import scrape_eliteserien_general_player_statistics_for_seasons

general_stats = scrape_eliteserien_general_player_statistics_for_seasons(
    seasons=[(2023, 2024)],
    delay_seconds=1,
    season_delay_seconds=2.0,
    timeout_seconds=90,
)

Done 1/240 | season=2024 | Odd vs Haugesund | players=40
Done 2/240 | season=2024 | Fredrikstad FK vs Bodø/Glimt | players=38
Done 3/240 | season=2024 | Lillestrøm vs Kristiansund BK | players=38


KeyboardInterrupt: 

In [13]:
def save_player_statistics_for_season(
    season: tuple[int, int | None],
    db_path: Path,
    delay_seconds: float = 0.1,
    season_delay_seconds: float = 0.1,  
    timeout_seconds: int = 30,
) -> None:
    """Scrape one Eliteserien season's xG/stats and store it in its own raw_match_statistics_<year> table."""
    year = season[1] or season[0]
    table_name = f"raw_player_statistics_{year}"


    data = scrape_eliteserien_general_player_statistics_for_seasons(
        seasons=[season],
        delay_seconds=delay_seconds,
        season_delay_seconds=season_delay_seconds,
        timeout_seconds=timeout_seconds,
    )
    

    if not data:
        print(f"No records returned for season {season}; skipping.")
        return

    df = pl.DataFrame(data).with_columns(pl.lit(datetime.now()).alias("ingested_at"))

    with duckdb.connect(str(db_path)) as connection:
        connection.register("new_data", df)
        connection.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM new_data")

    print(f"✓ Saved {len(data)} matches to {table_name}")

season_player_statistics = [(2020,2021)]

# Ingest xG/stats season by season, waiting 10s between seasons so one failing season doesn't block the rest
for index, season in enumerate(season_player_statistics):
    if index:
        sleep(10)
    try:
        save_player_statistics_for_season(season, delay_seconds = 3, db_path=db_path)
    except Exception as error:
        print(f"Skipping season {season}: {error}")


Skipping Bodø/Glimt vs Tromsø IL (2021-05-09): SofaScore API svarte med 403 for /api/v1/search/all?q=Bod%C3%B8%2FGlimt+Troms%C3%B8+IL.


KeyboardInterrupt: 

In [17]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - dim_teams
  - fct_league_standings
  - fct_matches
  - raw_eliteserien_results_2021
  - raw_eliteserien_results_2022
  - raw_eliteserien_results_2023
  - raw_eliteserien_results_2024
  - raw_eliteserien_results_2025
  - raw_eliteserien_results_2026
  - raw_match_statistics_2021
  - raw_match_statistics_2022
  - raw_match_statistics_2023
  - raw_match_statistics_2024
  - raw_match_statistics_2025
  - raw_match_statistics_2026


In [3]:
def hent_tabell_som_polars(table_name: str, db_path: str | Path = "data/brann.duckdb") -> pl.DataFrame:
    """
    Henter en tabell fra en DuckDB-database og returnerer den som en Polars DataFrame.

    Args:
        table_name: Navnet på tabellen som skal hentes.
        db_path: Sti til DuckDB-databasen (default: "data/brann.duckdb").

    Returns:
        Polars DataFrame med innholdet fra tabellen.
    """
    con = duckdb.connect(str(Path(db_path)))
    try:
        arrow_table = con.execute(f"SELECT * FROM {table_name}").arrow()
        df = pl.from_arrow(arrow_table)
    finally:
        con.close()
    return df

In [12]:
col_2023 = hent_tabell_som_polars("raw_match_statistics_2024").columns
col_2024 = hent_tabell_som_polars("raw_match_statistics_2025").columns


In [13]:
set(col_2024) - set(col_2023)

{'away_fouls_1st', 'away_fouls_2nd', 'home_fouls_1st', 'home_fouls_2nd'}

In [9]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("DROP TABLE IF EXISTS fct_match_statistics").fetchall()
con.close()

In [ ]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("select * from raw_eliteserien_results_2026").arrow()
pl.from_arrow(table_names)

date,matchday,home_team,away_team,result,report_url,snapshot_at,season,ingested_at
date,i64,str,str,str,str,"datetime[μs, Europe/Oslo]",i64,datetime[μs]
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
…,…,…,…,…,…,…,…,…
2026-09-20,22,"""Vålerenga""","""Fredrikstad FK""","""1:1""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-09-20,22,"""Sandefjord""","""IK Start""","""3:0""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546
2026-09-20,22,"""Tromsø IL""","""HamKam""","""3:2""","""https://www.transfermarkt.com/…",2026-09-22 09:41:41.565888 CEST,2026,2026-09-22 09:41:41.610546


: 